# VizEx — experimental analysis

Do the geometric features of stroboscopically induced visual hallucinations vary
with flicker frequency? Strobe trials only (`stimulus_type == LIGHT`).

For each feature: `feature ~ C(condition_hz)` with a random participant intercept,
an omnibus Wald test, Holm correction across the five features, and variance
components (trial-level SD, coefficient of variation, ICC).

Requires `015_feature_extraction` to have been run first.

In [ ]:
## imports

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm

# Only silence library churn. A blanket ignore would also hide the
# "Random effects covariance is singular" warnings, which matter here.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
## paths and style

PROJECT_PATH = Path().resolve().parent
TABLES_PATH = PROJECT_PATH / "015_tables"

TRIALS_FEATURES_CSV = TABLES_PATH / "vizex_features.csv"
TARGETS_FEATURES_CSV = TABLES_PATH / "vizex_targets_features.csv"

FEATURES = {
    "detail": "detail level",
    "r_directionality": "radial orientation",
    "theta_directionality": "tangential orientation",
    "brightness": "brightness",
    "contrast": "contrast",
}

LINE_WEIGHT_AXIS = 2.5
LINE_WEIGHT_MEAN = 3.5
FONT_SIZE_MAIN = 16
FONT_SIZE_TICKS = 11
MARKER_SIZE_DATA = 20
MARKER_ALPHA = 0.12
DEFAULT_COLOR = "#003366"

In [ ]:
## load the feature table

df_trials = pd.read_csv(TRIALS_FEATURES_CSV)
df_targets = pd.read_csv(TARGETS_FEATURES_CSV)

df_trials["condition_code"] = df_trials["condition_code"].astype(str)

# Guard against a stale or half-written table from 015.
uncropped = df_trials["png_filename"].str.contains("uncropped", na=False)
if uncropped.any():
    raise ValueError(f"{int(uncropped.sum())} uncropped rows present -- re-run 015.")

for feature in FEATURES:
    if feature not in df_trials.columns:
        raise KeyError(f"missing feature column '{feature}' -- re-run 015.")
    if df_trials[feature].isna().all():
        raise ValueError(f"feature column '{feature}' is all NaN -- re-run 015.")

print(f"trials: {len(df_trials):,} rows, {df_trials['participant_code'].nunique()} participants")
display(df_trials.head())

In [ ]:
## prepare the strobe-trial frame

df = df_trials[df_trials["stimulus_type"] == "LIGHT"].copy()

for col in list(FEATURES) + ["condition_hz"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Round away float noise so grouping by frequency is exact
df["condition_hz"] = df["condition_hz"].round(2)
df = df.dropna(subset=list(FEATURES) + ["condition_hz", "participant_code"]).copy()

# Ordered categorical over the frequencies actually present.
#
# NOTE: pd.Categorical(df["condition_hz"].sort_values()) would drop the index and
# assign POSITIONALLY, giving each row the i-th smallest frequency rather than its
# own. Always pass explicit categories.
HZ_LEVELS = sorted(df["condition_hz"].dropna().unique().tolist())
df["hz_cat"] = pd.Categorical(df["condition_hz"], categories=HZ_LEVELS, ordered=True)
assert (df["hz_cat"].astype(float) == df["condition_hz"]).all(), "hz_cat is misaligned"

print(f"{len(df):,} strobe trials, {df['participant_code'].nunique()} participants")
print(f"frequencies: {HZ_LEVELS}")
print(df.groupby("condition_hz").size().to_string())

In [ ]:
## plotting

def fmt_hz(hz) -> str:
    return f"{float(hz):.1f}".rstrip("0").rstrip(".")


def plot_feature_vs_frequency(feature, df, *, feature_name, y_label,
                              use_log_scale=False, y_limit_percentile=None,
                              use_robust_avg=False, show_error_bars=False,
                              force_y_min_0=False, line_color=DEFAULT_COLOR):
    """Raw trials as faint jittered dots, with the condition mean (or median) over the top."""
    rng = np.random.RandomState(42)
    jitter = 0.12

    hz_levels = list(df["hz_cat"].cat.categories)
    labels = [f"{fmt_hz(hz)} Hz" for hz in hz_levels]
    x_pos = np.arange(len(hz_levels), dtype=float)

    fig, ax = plt.subplots(figsize=(max(7.5, 0.6 * len(hz_levels)), 5.5))

    feature_data = df[feature].dropna()

    # A log axis needs strictly positive data
    if use_log_scale and (feature_data <= 0).any():
        positive = feature_data[feature_data > 0]
        floor = (positive.min() if positive.size else 1e-5) * 0.1
        feature_data = feature_data.clip(lower=floor)
        df = df.copy()
        df[feature] = df[feature].clip(lower=floor)

    if y_limit_percentile is not None:
        margin = (100.0 - y_limit_percentile) / 2.0
        y_min, y_max = np.percentile(feature_data, [margin, 100.0 - margin])
        if use_log_scale:
            ax.set_ylim(y_min * 0.8, y_max * 1.2)
        else:
            span = y_max - y_min
            ax.set_ylim(y_min - 0.02 * span, y_max + 0.02 * span)

    if force_y_min_0 and not use_log_scale:
        ax.set_ylim(bottom=0)

    for i, hz in enumerate(hz_levels):
        vals = df.loc[df["condition_hz"] == hz, feature].to_numpy(float)
        if vals.size == 0:
            continue
        ax.scatter(x_pos[i] + rng.normal(0, jitter, size=vals.size), vals,
                   s=MARKER_SIZE_DATA, color="black", alpha=MARKER_ALPHA,
                   edgecolors="none", zorder=1, clip_on=False)

    grouped = df.groupby("condition_hz")[feature]
    center = (grouped.median() if use_robust_avg else grouped.mean())
    center = center.reindex(hz_levels).to_numpy(float)

    if show_error_bars:
        err = grouped.sem().reindex(hz_levels).to_numpy(float)
        ax.errorbar(x_pos, center, yerr=err, fmt="none", ecolor=line_color,
                    elinewidth=LINE_WEIGHT_AXIS, capsize=4, zorder=2, clip_on=False)

    ax.plot(x_pos, center, linewidth=LINE_WEIGHT_MEAN, color=line_color, marker="o",
            markersize=7, markerfacecolor="white", markeredgewidth=2,
            zorder=3, clip_on=False)

    if use_log_scale:
        from matplotlib.ticker import ScalarFormatter
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(ScalarFormatter())

    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_linewidth(LINE_WEIGHT_AXIS)

    ax.set_title(f"{feature_name} vs stroboscopic frequency", fontsize=FONT_SIZE_MAIN,
                 fontweight="bold", loc="left", pad=15)
    ax.set_ylabel(y_label, fontsize=FONT_SIZE_MAIN, fontweight="bold", labelpad=12)
    ax.set_xlabel("stroboscopic frequency", fontsize=FONT_SIZE_MAIN,
                  fontweight="bold", labelpad=12)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=45, ha="right",
                       fontsize=FONT_SIZE_TICKS, fontweight="bold")
    ax.tick_params(axis="both", width=LINE_WEIGHT_AXIS, labelsize=FONT_SIZE_TICKS)

    plt.tight_layout()
    plt.show()

## Main experimental analysis

In [ ]:
results, variance_export = [], []

for feature, name in FEATURES.items():
    plot_feature_vs_frequency(feature, df, feature_name=name, y_label=name)

    try:
        model = smf.mixedlm(f"{feature} ~ C(condition_hz)", df, groups=df["participant_code"])
        result = model.fit(reml=True)

        table = result.wald_test_terms().table
        chi2 = float(np.squeeze(table.loc["C(condition_hz)", "statistic"]))
        df_num = int(np.squeeze(table.loc["C(condition_hz)", "df_constraint"]))
        p_unc = float(np.squeeze(table.loc["C(condition_hz)", "pvalue"]))

        var_resid = float(result.scale)
        sd_resid = float(np.sqrt(var_resid))
        var_participant = float(result.cov_re.iloc[0, 0])
        icc = var_participant / (var_participant + var_resid)

        grand_mean = float(df[feature].mean())
        trial_cv = (sd_resid / grand_mean) * 100 if grand_mean != 0 else 0.0

        # Flag rather than silently report an unidentified participant variance.
        singular = bool(not np.isfinite(var_participant) or var_participant < 1e-10)
        if singular:
            print(f"  WARNING [{name}]: random-effects variance ~0 ({var_participant:.3g}); "
                  f"ICC and DT_plus_HT are unidentified.")

        results.append({
            "Feature": name, "Mean": grand_mean, "Chi2": chi2, "df": df_num,
            "p_unc": p_unc, "Trial_SD": sd_resid, "Trial_CV%": trial_cv, "ICC": icc,
        })
        variance_export.append({
            "Feature": feature,
            "Signal_Exp": float(np.var(model.predict(result.params))),
            "DT_plus_HT": var_participant,
            "DN_plus_HN": var_resid,
            "RE_singular": singular,
        })

    except Exception as e:
        print(f"Failed to fit model for {name}: {e}")

pd.DataFrame(variance_export).to_csv(TABLES_PATH / "experimental_results_full.csv", index=False)

In [ ]:
## Holm correction and summary table

res_df = pd.DataFrame(results)
_, p_holm, _, _ = multipletests(res_df["p_unc"], alpha=0.05, method="holm")
res_df["p_holm"] = p_holm

print("\n" + "=" * 85)
print("STATISTICAL SUMMARY: TRIAL-LEVEL NOISE VS. TREND SIGNIFICANCE")
print("=" * 85)

disp = res_df.copy()
disp["Mean"] = disp["Mean"].round(2)
disp["Chi2"] = disp["Chi2"].round(1)
disp["Trial_SD"] = disp["Trial_SD"].round(2)
disp["Trial_CV%"] = disp["Trial_CV%"].map("{:.1f}%".format)
disp["ICC"] = disp["ICC"].round(3)
disp["p_holm"] = disp["p_holm"].apply(lambda x: "< .001" if x < 0.001 else f"{x:.4f}")

print(disp[["Feature", "Mean", "Chi2", "df", "p_holm",
            "Trial_SD", "Trial_CV%", "ICC"]].to_string(index=False))

print("\nInterpretation Note: High Trial_SD with low ICC indicates variance is driven by")
print("stochastic trial-level noise, not stable participant differences.")

## Follow-up: 10 Hz vs 15 Hz (detail)

In [ ]:
def run_detail_10_vs_15():
    """Trial-level LME contrasting the two adjacent frequencies of interest.
    10 Hz is the reference level, so the coefficient is (15 Hz - 10 Hz)."""
    d = df[df["condition_hz"].isin([10, 15])].copy()
    d = d.dropna(subset=["detail", "condition_hz", "participant_code"])
    d["hz_cat"] = pd.Categorical(d["condition_hz"], categories=[10, 15], ordered=True)

    print(f"Total Trials: {len(d)}")
    print(f"Unique Participants: {d['participant_code'].nunique()}")

    m = smf.mixedlm("detail ~ C(hz_cat)", d, groups=d["participant_code"]).fit(reml=False)

    print("\n================ LME RESULTS (10 Hz vs 15 Hz) ================\n")
    print(m.summary())

    coef_keys = [k for k in m.params.keys() if "T.15" in k]
    if coef_keys:
        k = coef_keys[0]
        z = m.params[k] / m.bse[k]
        p = 2 * norm.sf(abs(z))
        p_str = "p < .001" if p < 0.001 else f"p = {p:.3f}".replace("0.", ".", 1)
        print(f"\nAPA Reporting Fragment (Detail):\n(z = {z:.2f}, {p_str})\n")

    plot_feature_vs_frequency("detail", d, feature_name="Detail Level",
                              y_label="Normalised detail level", show_error_bars=True)


run_detail_10_vs_15()